In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load the preprocessed dataset saved from your EDA notebook
df_clean = pd.read_csv('/content/ted_dataset/ted_talks_preprocessed.csv')

print("df_clean shape:", df_clean.shape)
print("Number of unique titles:", df_clean["title"].nunique())
print("Number of unique talk IDs:", df_clean["talk_id"].nunique())

print("\nColumns:")
print(df_clean.columns.tolist())

df_clean shape: (4005, 33)
Number of unique titles: 4005
Number of unique talk IDs: 4005

Columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'transcript_words', 'topic_count', 'title_word_count', 'description_word_count', 'about_speakers_word_count', 'transcript_word_count', 'duration_minutes', 'log_views', 'related_talk_count', 'title_clean', 'topics_clean', 'description_clean', 'transcript_clean', 'combined_text']


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use the current 4,005-talk dataset
text_data = df_clean["combined_text"].fillna("")

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# Fit and transform
tfidf_matrix = tfidf.fit_transform(text_data)

print("TF-IDF Matrix successfully created!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF Matrix successfully created!
Matrix shape: (4005, 5000)


In [7]:

from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all TED Talks
cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine similarity matrix created!")
print("Matrix shape:", cosine_sim.shape)

Cosine similarity matrix created!
Matrix shape: (4005, 4005)


In [8]:
# Map each title to its dataframe index
indices = pd.Series(
    df_clean.index,
    index=df_clean["title"]
).drop_duplicates()

print("Title-index mapping created!")

Title-index mapping created!


In [10]:
def recommend_talks(title, num_recommendations=5):

    if title not in indices:
        print("Talk not found.")
        return

    # Get index of selected talk
    idx = indices[title]

    # Get similarity scores
    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    # Sort from highest to lowest similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected talk itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != idx
    ]

    # Select top recommendations
    top_scores = similarity_scores[:num_recommendations]

    recommended_indices = [
        item[0] for item in top_scores
    ]

    recommendations = df_clean.iloc[
        recommended_indices
    ][
        ["title", "speaker_1", "views", "url"]
    ].copy()

    recommendations["similarity_score"] = [
        item[1] for item in top_scores
    ]

    return recommendations

In [11]:
print(df_clean["title"].head(10).tolist())

['Averting the climate crisis', "The best stats you've ever seen", 'Simplicity sells', 'Greening the ghetto', 'Do schools kill creativity?', "Behind the design of Seattle's library", 'Letting go of God', "Let's teach religion -- all religion -- in schools", 'A life of purpose', 'My wish: A global day of film']


In [12]:
recommend_talks(
    df_clean["title"].iloc[0],
    num_recommendations=5
)

,title,speaker_1,views,url,similarity_score
209,New thinking on the climate crisis,Al Gore,2187523,https://www.ted.com/talks/al_gore_new_thinking...,0.367240
3768,How we can turn the tide on climate,Chris Anderson,1493370,https://www.ted.com/talks/christiana_figueres_...,0.355222
555,A one-man world summit,Rory Bremner,1207951,https://www.ted.com/talks/rory_bremner_a_one_m...,0.340232
2398,A climate solution where all sides can win,Ted Halstead,1566306,https://www.ted.com/talks/ted_halstead_a_clima...,0.338672
3944,How we could change the planet's climate future,David Wallace-Wells,1062677,https://www.ted.com/talks/david_wallace_wells_...,0.337869


In [13]:
recommend_talks(
    df_clean["title"].iloc[500],
    num_recommendations=5
)

,title,speaker_1,views,url,similarity_score
706,My mind-shifting Everest swim,Lewis Pugh,1207720,https://www.ted.com/talks/lewis_pugh_my_mind_s...,0.516614
37,Why did I ski to the North Pole?,Ben Saunders,924205,https://www.ted.com/talks/ben_saunders_why_did...,0.514525
1074,Extreme swimming with the world's most dangero...,Diana Nyad,748096,https://www.ted.com/talks/diana_nyad_extreme_s...,0.418783
3916,The Arctic vs. the Antarctic,Camille Seaman,2724244,https://www.ted.com/talks/camille_seaman_the_a...,0.387043
3069,The dangerous race for the South Pole,Elizabeth Leane,218825,https://www.ted.com/talks/elizabeth_leane_the_...,0.376486


In [19]:
def user_recommendation():

    user_input = input(
        "Enter a TED Talk title or keyword: "
    ).strip().lower()

    # Search titles using partial matching
    matches = df_clean[
        df_clean["title"]
        .str.lower()
        .str.contains(user_input, na=False)
    ]

    # No matches
    if len(matches) == 0:
        print("\n No TED Talks found for:", user_input)
        print("Try another keyword.")
        return

    # Show matching talks
    print("\nMatching TED Talks:\n")

    display(
        matches[
            ["title", "speaker_1", "views"]
        ].head(10)
    )

    # If multiple talks are found, let user select one
    if len(matches) > 1:

        print("\nSelect a talk:")

        for i, title in enumerate(
            matches["title"].head(10),
            start=1
        ):
            print(f"{i}. {title}")

        choice = int(
            input("\nEnter the number of the talk: ")
        )

        selected_title = matches[
            "title"
        ].iloc[choice - 1]

    else:
        selected_title = matches["title"].iloc[0]

    # Generate recommendations
    print("\nSelected TED Talk:")
    print(selected_title)

    print("\n Recommended TED Talks:")

    recommendations = recommend_talks(
        selected_title,
        num_recommendations=5
    )

    display(recommendations)


In [ ]:
user_recommendation()

Enter a TED Talk title or keyword: climate crisis

Matching TED Talks:



,title,speaker_1,views
0,Averting the climate crisis,Al Gore,3523392
209,New thinking on the climate crisis,Al Gore,2187523



Select a talk:
1. Averting the climate crisis
2. New thinking on the climate crisis
